# A toy language model

Everything in this notebook is built from layers you have already met:
an embedding table, positional encoding, causal transformer blocks and a
dense projection. Nothing is imported from outside `si`, and the whole thing
is NumPy.

**What to expect.** This trains on a few thousand characters, on a CPU, in
pure NumPy. The loss will fall clearly and the samples will pick up the
*statistics* of the text: plausible letter pairs, word-like runs, line breaks
in roughly the right places, a Shakespearean cadence. It will **not** produce
fluent English. That gap is compute and data, not a flaw in the code -- the
mechanism here is the same one used at scale.


In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt

from si.data import Dataset
from si.supervised.nn import (CharTokenizer, make_windows, load_text,
                             build_language_model, generate)

## What attention actually computes

Before assembling a model, look at the mechanism on its own. `SelfAttention`
takes a batch of sequences, shape `(examples, positions, features)`, and
returns the same shape -- but every position's output is now a weighted
average of *all* the positions it is allowed to see.

Those weights are the interesting part. They are computed from the data, one
distribution per position, and we can look at them directly.


In [ ]:
from si.supervised.nn import SelfAttention, LayerNorm, SGD

sentence = 'to be or not to be'
tokens = sentence.split()

np.random.seed(0)
attention = SelfAttention(d_model=8)
attention.initialize(SGD())

# One random embedding per token, then attend.
x = np.random.randn(1, len(tokens), 8)
out = attention.forward(x)
print('in ', x.shape, ' -> out', out.shape)
print('attention weights:', attention.weights.shape, '= (examples, positions, positions)')

In [ ]:
weights = attention.weights[0]

fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(weights, cmap='viridis', vmin=0)
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel('attends to'); ax.set_ylabel('position')
ax.set_title('untrained attention weights')
fig.colorbar(image); fig.tight_layout(); plt.show()

print('every row sums to 1:', np.allclose(weights.sum(axis=1), 1))

Untrained, the weights are near-uniform -- the layer has learned nothing, so
it averages everything roughly equally. What matters is the *shape* of the
object: one row per position, each a probability distribution over the
positions it may attend to.

### The causal mask

For a language model the mask is not optional. Predicting the next character
is only a task if the model cannot look at it. `causal=True` forbids attending
forward, which shows up as an exactly-zero upper triangle.


In [ ]:
causal = SelfAttention(d_model=8, causal=True)
causal.initialize(SGD())
causal.forward(x)

fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(causal.weights[0], cmap='viridis', vmin=0)
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel('attends to'); ax.set_ylabel('position')
ax.set_title('causal: no looking ahead')
fig.colorbar(image); fig.tight_layout(); plt.show()

print('upper triangle is exactly zero:',
      np.allclose(np.triu(causal.weights[0], k=1), 0))
print('first position can only see itself:', round(causal.weights[0, 0, 0], 6))

In [ ]:
# Proof rather than illustration: change the LAST token and every earlier
# output must be untouched.
disturbed = x.copy()
disturbed[0, -1, :] += 100.0
before, after = causal.forward(x), causal.forward(disturbed)
for position, token in enumerate(tokens):
    unchanged = np.allclose(before[0, position], after[0, position], atol=1e-10)
    print(f'  position {position} ({token:4}) unchanged: {unchanged}')

### LayerNorm, and why not BatchNorm

Transformers normalise over the FEATURE axis, per position. `BatchNormalization`
normalises each feature over the BATCH instead. The axis is the whole
difference: LayerNorm needs no running statistics, behaves identically at
training and inference, and works with a batch of one -- which is exactly the
situation when generating text one character at a time.


In [ ]:
norm = LayerNorm(8)
norm.initialize(SGD())
normalised = norm.forward(x)

print('per-position mean over features:', np.round(normalised[0].mean(axis=-1), 8))
print('per-position std  over features:', np.round(normalised[0].std(axis=-1), 4))
print()
single = norm.forward(np.random.randn(1, 1, 8))
print('a batch of one still normalises:', np.isfinite(single).all(),
      '| mean', round(float(single.mean()), 8))

### Several heads at once

One head produces one distribution per position, so it can express one
relationship at a time. `MultiHeadAttention` splits the features into `h`
slices and attends independently over each, so `h` relationships can be
attended to simultaneously.

It costs no more: `h` heads of width `d_model/h` do the same total work as one
head of width `d_model`. The heads are carved from the same projections by
reshaping, so the split adds no parameters. This is what `TransformerBlock`
uses internally.


In [ ]:
from si.supervised.nn import MultiHeadAttention

multi = MultiHeadAttention(d_model=8, n_heads=4, causal=True)
multi.initialize(SGD())
multi.forward(x)
print(multi)
print('weights:', multi.weights.shape, '= (examples, heads, positions, positions)')
print('each head splits d_model 8 into d_k =', multi.d_k)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for head, ax in enumerate(axes):
    ax.imshow(multi.weights[0, head], cmap='viridis', vmin=0)
    ax.set_title(f'head {head}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('four causal heads, each free to specialise')
fig.tight_layout(); plt.show()

print('every head is causal:',
      all(np.allclose(np.triu(multi.weights[0, h], k=1), 0) for h in range(4)))

## The corpus

A few kilobytes of public-domain Shakespeare. Character-level modelling needs
no tokeniser training and has no out-of-vocabulary problem: the vocabulary is
simply the set of characters that occur.


In [ ]:
DIR = os.path.dirname(os.path.realpath('.'))
text = load_text(os.path.join(DIR, 'datasets/tiny-text.txt'))
print(f'{len(text)} characters')
print(text[:280])

In [ ]:
tokenizer = CharTokenizer(text)
print(tokenizer)
print('vocabulary:', repr(''.join(tokenizer.chars)))
print()
print('encode/decode round-trips:', tokenizer.decode(tokenizer.encode(text)) == text)

## Training windows

The task is next-character prediction, so the target is the input shifted one
step left. Because the blocks are **causal**, position *i* may only attend to
positions up to *i* -- so a single window of length 32 gives 32 separate
prediction problems, each with a different amount of context.

Without the causal mask the task would be trivial: the answer for position *i*
sits at position *i+1* of the same input.


In [ ]:
SEQ_LEN = 32
ids = tokenizer.encode(text)
X, y = make_windows(ids, SEQ_LEN, stride=3)
print(f'{len(X)} windows of {SEQ_LEN}')
print()
print('X[0]:', repr(tokenizer.decode(X[0])))
print('y[0]:', repr(tokenizer.decode(y[0])))

## The model

Two transformer blocks. `TransformerBlock` uses post-norm, as published, which
trains cleanly to about four blocks and then degrades sharply -- see its
docstring for the measured numbers. Two is comfortably inside that range.

The final `Dense` produces one score per vocabulary character at every
position, which `softmax-cross-entropy` consumes directly.


In [ ]:
from si.supervised.nn import (NN, Embedding, PositionalEncoding,
                             TransformerBlock, Dense, Adam)

# Written out rather than via build_language_model, so the stack is visible.
# build_language_model(...) is a shortcut for exactly this.
np.random.seed(0)
D_MODEL, N_HEADS, N_BLOCKS, EPOCHS = 64, 4, 2, 40

model = NN(epochs=EPOCHS, batch_size=64, verbose=True, step=4,
           loss='softmax-cross-entropy', optimizer=Adam(0.005))
model.add(Embedding(tokenizer.vocab_size, D_MODEL))
model.add(PositionalEncoding(SEQ_LEN, D_MODEL))
for _ in range(N_BLOCKS):
    model.add(TransformerBlock(D_MODEL, N_HEADS, causal=True))
model.add(Dense(D_MODEL, tokenizer.vocab_size))

print(model)

## Training

A useful reference point: a model that has learned nothing outputs a uniform
distribution, giving a cross entropy of `ln(vocab_size)`. Anything below that
means the model has learned something about the text.


In [ ]:
model.fit(Dataset(X, y))

In [ ]:
losses = [model.history[e][0] for e in sorted(model.history)]
baseline = np.log(tokenizer.vocab_size)

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(losses) + 1), losses, label='training loss')
plt.axhline(baseline, linestyle='--', color='grey',
            label=f'uniform guess = ln({tokenizer.vocab_size}) = {baseline:.2f}')
plt.xlabel('epoch'); plt.ylabel('cross entropy'); plt.legend(); plt.tight_layout()
plt.show()

accuracy = (model.predict(X).argmax(axis=-1) == y).mean()
print(f'loss {losses[0]:.3f} -> {losses[-1]:.3f}   (uniform baseline {baseline:.3f})')
print(f'next-character accuracy on the training windows: {accuracy:.3f}')

## Generation

Autoregressive generation is just the training task run in a loop: predict the
next character, append it, predict again. Only the **last** position's logits
are used -- the others describe characters we already have.

`temperature` reshapes the distribution before sampling:

- `0` is greedy -- always the single most likely character. Deterministic, and
  prone to falling into loops.
- `< 1` sharpens: safer, more repetitive.
- `1` samples from the model's own distribution.
- `> 1` flattens towards uniform: more surprising, less coherent.


In [ ]:
prompt = 'To be, or not to be'

for temperature in (0.0, 0.5, 0.8, 1.0):
    label = 'greedy' if temperature == 0 else f'temperature {temperature}'
    print(f'--- {label} ---')
    print(generate(model, tokenizer, prompt, n_chars=200,
                   seq_len=SEQ_LEN, temperature=temperature,
                   random_state=0))
    print()

## What the model actually learned

Read the samples for *structure* rather than meaning. Even at this scale the
model has picked up:

- spelling patterns -- which letters follow which, so most runs are
  pronounceable and many are real words;
- word boundaries, and roughly plausible word lengths;
- line structure, breaking lines at sensible intervals;
- capitalisation after newlines, and apostrophes inside words.

None of that was programmed. It came from the same next-character objective,
the same attention mechanism, and the same gradients used at scale -- there is
no difference in kind between this and a large language model, only in size
and in how much text it has read.

Two things to try:

1. `n_blocks=1` versus `n_blocks=4`, and watch how much depth buys here.
2. `stride=1` in `make_windows` for three times the training data.
